In [ ]:
# -*- coding: utf-8 -*-
"""
Tri-modelový skript s failover logikou (bez RAG).
- Skúsi primary -> ak zlyhá, secondary -> ak zlyhá, tertiary.
- Tezaurus + regex matchovanie, DOCX/RTF sekcie, batch inference.
- Výstup: CSV (MODE = 'title' alebo 'keyword'), obsahuje stĺpec `model_used`.

Optimalizované pre ~12 GB VRAM (4-bit pre väčšie modely + sekvenčné načítanie).
"""

import os
os.environ["OMP_NUM_THREADS"] = "12"
os.environ["MKL_NUM_THREADS"] = "12"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

import re
import unicodedata
from datetime import datetime
from typing import List, Tuple, Dict, Any, Optional

import warnings
import torch
import pandas as pd

try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from docx import Document
from striprtf.striprtf import rtf_to_text

# ------------------------------
# Režim
# ------------------------------
MODE = 'title'  # 'title' alebo 'keyword'

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'

THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'
STOPWORDS_TXT  = './Input/stopword_dictionary.txt'

# Failover poradie (1 -> 2 -> 3)
MODELS = [
    dict(
        name='tertiary:marcuscedricridia/8B-Nemotaur-IT',
        hf_id='marcuscedricridia/8B-Nemotaur-IT',
        quant4bit=True,
        dtype='fp16',
        gen=dict(max_new_tokens=128, do_sample=True, temperature=0.2, top_p=0.9, repetition_penalty=1.6),
        batch_size=4
    ),
    dict(
        name='primary:Qwen/Qwen2.5-3B-Instruct',
        hf_id='Qwen/Qwen2.5-3B-Instruct',
        quant4bit=True,
        dtype='fp16',
        gen=dict(max_new_tokens=128, do_sample=True, temperature=0.2, top_p=0.9, repetition_penalty=1.4),
        batch_size=6
    ),
    dict(
        name='secondary:google/gemma-2-2b-it',
        hf_id='google/gemma-2-2b-it',
        quant4bit=False,
        dtype='bf16',  # ak GPU nepodporí bf16, prepne sa na fp16
        gen=dict(max_new_tokens=128, do_sample=True, temperature=0.2, top_p=0.9, repetition_penalty=1.5),
        batch_size=6
    ),
]

MAX_TEXT_CHARS = 8000
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# ------------------------------
# Stopwords / Tezaurus
# ------------------------------
def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f'[STOP] Loaded {len(words)} stopwords from {path}')
        return words
    except FileNotFoundError:
        print(f'[STOP] No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'[STOP] Failed to load stopwords: {e}')
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))
    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]
    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    pq_path  = os.path.splitext(xlsx_path)[0] + ".parquet"
    csv_path = os.path.splitext(xlsx_path)[0] + ".csv"
    if HAS_POLARS and os.path.exists(pq_path):
        df = pl.read_parquet(pq_path)
        raw = df[column].drop_nulls().cast(pl.Utf8).to_list() if column in df.columns else []
    elif HAS_POLARS and os.path.exists(csv_path):
        df = pl.read_csv(csv_path)
        raw = df[column].drop_nulls().cast(pl.Utf8).to_list() if column in df.columns else []
    else:
        if not os.path.exists(xlsx_path):
            raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
        df = pd.read_excel(xlsx_path, engine='openpyxl')
        if column not in df.columns:
            raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
        raw = df[column].dropna().astype(str).tolist()

    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', str(cell)))

    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t: continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)

    print(f'[TH] Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    if not text: return []
    hits, t_na = {}, strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0: hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

# ------------------------------
# DOCX / RTF sekcionovanie
# ------------------------------
_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None: buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None: buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# ------------------------------
# Prompting
# ------------------------------
def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars: return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def prompt_title(text: str, th_priority: List[str], th_sample: List[str]) -> str:
    pri = "\n".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:40])
    return (
        "ÚLOHA: Navrhni JEDEN presný a zrozumiteľný právny nadpis (3–12 slov) k textu nižšie.\n"
        "Požiadavky: vecný, bez bodky na konci, žiadne úvodzovky. Použi slovenčinu.\n"
        "Vráť len samotný nadpis, nič iné.\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "NADPIS:"
    )

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str]) -> str:
    pri = ", ".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:40])
    return (
        "ÚLOHA: Vyber JEDEN najvhodnejší právny pojem (1–4 slová) k nasledujúcemu textu.\n"
        "Preferuj pojmy z tezauru. Vráť len samotný pojem, bez vysvetlenia a bez bodky. Použi slovenčinu.\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "POJEM:"
    )

# ------------------------------
# Robustné parsovanie výstupov
# ------------------------------
def _normalize_output_item(item) -> str:
    if item is None:
        raw = ""
    elif isinstance(item, dict):
        raw = item.get("generated_text") or item.get("text") or item.get("summary_text") or ""
    elif isinstance(item, list):
        if item and isinstance(item[0], dict):
            raw = item[0].get("generated_text") or item[0].get("text") or item[0].get("summary_text") or ""
        else:
            raw = " ".join([_normalize_output_item(x) for x in item])
    else:
        raw = str(item)
    txt = raw or ""
    txt = re.sub(r'^[\-\•\:\s]+', '', txt).strip()
    txt = re.sub(r'[\.，。…]+$', '', txt).strip()
    txt = txt.strip(' "„”')
    lines = [ln for ln in txt.splitlines() if ln.strip()]
    return lines[0].strip() if lines else ""

def safe_llm_call_batch(
    pipe, inputs: List[str],
    *, batch_size: int, num_return_sequences: int,
    max_new_tokens: int,
    temperature: Optional[float]=None, top_p: Optional[float]=None,
    max_retries: int = 2
):
    bs  = max(1, batch_size)
    nrs = max(1, num_return_sequences)
    kwargs = dict(batch_size=bs, num_return_sequences=nrs, max_new_tokens=max_new_tokens, return_full_text=False)
    if temperature is not None: kwargs["temperature"] = temperature
    if top_p is not None: kwargs["top_p"] = top_p
    for attempt in range(max_retries + 1):
        try:
            out = pipe(inputs, **kwargs)
            return out, bs, nrs
        except RuntimeError as e:
            msg = str(e)
            if "CUDA out of memory" in msg or "device-side assert" in msg:
                torch.cuda.empty_cache()
                if bs > 1: bs = max(1, bs // 2)
                if nrs > 1: nrs = 1
                kwargs["batch_size"] = bs
                kwargs["num_return_sequences"] = nrs
                kwargs["max_new_tokens"] = max(64, max_new_tokens // 2)
                continue
            raise
    return [], bs, nrs

def batched_generate_texts(pipe, prompts: List[str], batch_size: int, gen_kwargs: Dict[str, Any]) -> List[str]:
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out, used_bs, used_nrs = safe_llm_call_batch(
            pipe, batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=1,
            max_new_tokens=gen_kwargs.get("max_new_tokens", 128),
            temperature=gen_kwargs.get("temperature"),
            top_p=gen_kwargs.get("top_p"),
        )
        for item in out:
            outputs.append(_normalize_output_item(item))
    if len(outputs) != len(prompts):
        print(f"[WARN] outputs={len(outputs)} != inputs={len(prompts)} (doplním prázdne)")
        while len(outputs) < len(prompts): outputs.append("")
        if len(outputs) > len(prompts): outputs = outputs[:len(prompts)]
    return outputs

# ------------------------------
# Načítanie pipeline
# ------------------------------
def load_textgen_pipeline(hf_id: str, *, quant4bit: bool, dtype: str, gen_params: Dict[str, Any]):
    dtype_map = {'fp16': torch.float16, 'bf16': torch.bfloat16, 'fp32': torch.float32}
    use_dtype = dtype_map.get(dtype, torch.float16 if torch.cuda.is_available() else torch.float32)
    device_map = {"": DEVICE} if torch.cuda.is_available() else "cpu"
    qconf = None
    if quant4bit:
        qconf = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=use_dtype if use_dtype != torch.float32 else torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        device_map=device_map,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconf,
        dtype=None if qconf is not None else use_dtype,
    )
    tok = AutoTokenizer.from_pretrained(hf_id)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    pipe = pipeline("text-generation", model=model, tokenizer=tok, device_map=device_map, return_full_text=False, **gen_params)
    return pipe

# ------------------------------
# Failover beh: vráti prvý úspešný výsledok
# ------------------------------
def run_with_fallback(jobs: List[Tuple[str, str, str, List[str], str]]) -> Tuple[List[str], str]:
    """
    Skúsi modely v poradí MODELS. Po prvom úspechu vráti (generations, model_name).
    Úspech = podarí sa spracovať všetky prompty (alebo aspoň rovnaký počet výstupov).
    """
    prompts = [j[-1] for j in jobs]
    for mcfg in MODELS:
        name = mcfg["name"]
        print(f"\n[LLM] Loading {name} ...")
        pipe = None
        try:
            pipe = load_textgen_pipeline(mcfg["hf_id"], quant4bit=mcfg["quant4bit"], dtype=mcfg["dtype"], gen_params=mcfg["gen"])
            print(f"[LLM] Loaded {name}")
            gens = batched_generate_texts(pipe, prompts, batch_size=mcfg["batch_size"], gen_kwargs=mcfg["gen"])
            if len(gens) != len(prompts):
                raise RuntimeError(f"Generated {len(gens)} != inputs {len(prompts)}")
            # cleanup a návrat
            del pipe
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            return gens, name
        except Exception as e:
            print(f"[LLM] {name} failed: {e}")
        finally:
            if pipe is not None:
                try:
                    del pipe
                except Exception:
                    pass
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
    # Ak zlyhajú všetky modely:
    print("[LLM] All models failed – returning empty outputs.")
    return ["" for _ in prompts], "none"

# ------------------------------
# Driver
# ------------------------------
def process_all(input_dir=INPUT_DIR):
    files = sorted(os.listdir(input_dir), key=str.lower)
    jobs: List[Tuple[str, str, str, List[str], str]] = []  # (mode, file, header, prio_terms, prompt)

    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f'No text in {f}'); continue

        for header, text in secs:
            if not text.strip(): continue
            short = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:40]
            if MODE == 'title':
                prompt = prompt_title(short, prio_terms, sample_terms)
                jobs.append(("title", f, header, prio_terms, prompt))
            else:
                prompt = prompt_keyword(short, prio_terms, sample_terms)
                jobs.append(("keyword", f, header, prio_terms, prompt))
        print(f'Queued {f} with {len(secs)} sections.')

    if not jobs:
        print('No jobs queued.')
        return pd.DataFrame([])

    # Failover run
    gens, model_used = run_with_fallback(jobs)

    # Kompozícia riadkov
    rows = []
    for (mode, f, header, prio_terms, _), gen in zip(jobs, gens):
        if mode == 'title':
            rows.append({
                "model_used": model_used,
                "file": f, "header": header,
                "suggested_title": gen,
                "method": "llm-only",
                "priority_terms": "; ".join(prio_terms[:20])
            })
        else:
            rows.append({
                "model_used": model_used,
                "file": f, "header": header,
                "top_keyword": gen,
                "method": "llm-only",
                "priority_terms": "; ".join(prio_terms[:20])
            })

    # Export
    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if MODE == 'title':
        df = pd.DataFrame(rows, columns=["model_used","file","header","suggested_title","method","priority_terms"])
        stub = "llm_only_titles"
    else:
        df = pd.DataFrame(rows, columns=["model_used","file","header","top_keyword","method","priority_terms"])
        stub = "llm_only_keywords"
    csv_path = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")
    return df

# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    torch.set_num_threads(12)
    warnings.filterwarnings("ignore", message=r".*Some weights.*not of the same dtype.*")
    warnings.filterwarnings("ignore", message=r".*Tokenizer parallelism.*")

    df = process_all(INPUT_DIR)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))


[STOP] Loaded 76 stopwords from ./Input/stopword_dictionary.txt
[TH] Thesaurus terms loaded: 3000
